# Laboratorium 6

Celem szóstego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmu głębokiego uczenia aktywnego - REINFORCE. Zaimplementowany algorytm będzie testowany z wykorzystaniem środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [1]:
from collections import deque
import gym
import numpy as np
import random

if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Dołączenie bibliotek do obsługi sieci neuronowych

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical


class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_sizes=(128, 64)):
        super().__init__()
        h1, h2 = hidden_sizes
        self.net = nn.Sequential(
            nn.Linear(state_size, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, action_size),
            nn.Softmax(dim=-1),
        )

    def forward(self, x):
        return self.net(x)

    def predict(self, state):
        state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            probs = self.forward(state_tensor).squeeze(0)
        return probs

## Zadanie 1 - REINFORCE

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu REINFORCE. Wagi sieci aktualizowane są zgodnie ze wzorem:
\begin{equation*}
    \theta \leftarrow \theta + \alpha G_t \nabla_\theta log \pi_{\theta}(a_t, s_t | \theta)
\end{equation*}.
</p>

In [ ]:
class REINFORCEAgent:
    def __init__(self, state_size, action_size, model):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = 0.99    # discount rate
        self.learning_rate = 0.001
        self.model = model
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.state_memory = []
        self.action_memory = []
        self.reward_memory = []
        
        
    def remember(self, state, action, reward):
        #Function adds information to the memory about last action and its results
        self.state_memory.append(state)
        self.action_memory.append(action)
        self.reward_memory.append(reward)

    def get_action(self, state):
        """
        Compute the action to take in the current state, basing on policy returned by the network.

        Note: To pick action according to the probability generated by the network
        """

        #
        # INSERT CODE HERE to get action in a given state
        predictions = self.model.predict(state)
        predictions = predictions.detach().cpu().numpy()
        chosen_action = np.random.choice(self.action_size, p=predictions)
        #        
        
        return chosen_action

    def count_cumulative_rewards(self, rewards, gamma=0.99):
        cumulative_rewards = np.empty(len(rewards), dtype=np.float32)
        accumulate_reward = 0.0

        for idx in range(len(rewards) - 1, -1, -1):
            accumulate_reward = rewards[idx] + gamma * accumulate_reward
            cumulative_rewards[idx] = accumulate_reward

        return cumulative_rewards

    def replay(self, batch_size=None):
        """
        Function learn network using data stored in state, action and reward memory. 
        First calculates G_t for each state and train network
        """
        states = torch.as_tensor(np.array(self.state_memory), dtype=torch.float32)
        actions = torch.as_tensor(self.action_memory, dtype=torch.int64)
        returns = self.count_cumulative_rewards(self.reward_memory, gamma=self.gamma)
        returns = torch.as_tensor(returns, dtype=torch.float32)

        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        probs = self.model(states)
        dist = Categorical(probs=probs) #rozkład prawdopodobieństwa akcji w stanie 
        log_probs = dist.log_prob(actions) # log prawdopodobieństwo wykonanych akcji
        loss = -(log_probs * returns).sum()

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        self.state_memory = []
        self.action_memory = []
        self.reward_memory = []


Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [4]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
learning_rate = 0.001

model = PolicyNetwork(state_size, action_size)
agent = REINFORCEAgent(state_size, action_size, model)

d:\.Astudia\.venv\Lib\site-packages\gym\envs\registration.py:555: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(


Przygotuj funkcję obliczającą wartość nagrody skumulowanej:

In [5]:
def get_cumulative_rewards(rewards,  # rewards at each step
                           gamma=0.99  # discount for reward
                           ):
    """
    based on https://github.com/yandexdataschool/Practical_RL/blob/spring20/week06_policy_based/reinforce_tensorflow.ipynb
    take a list of immediate rewards r(s,a) for the whole session
    compute cumulative rewards R(s,a) (a.k.a. G(s,a) in Sutton '16)
    R_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...

    The simple way to compute cumulative rewards is to iterate from last to first time tick
    and compute R_t = r_t + gamma*R_{t+1} recurrently

    You must return an array/list of cumulative rewards with as many elements as in the initial rewards.
    """
    cumulative_rewards = agent.count_cumulative_rewards(rewards, gamma) 

    return cumulative_rewards


assert len(get_cumulative_rewards(range(100))) == 100
assert np.allclose(get_cumulative_rewards([0, 0, 1, 0, 0, 1, 0], gamma=0.9),
                   [1.40049, 1.5561, 1.729, 0.81, 0.9, 1.0, 0.0])
assert np.allclose(get_cumulative_rewards([0, 0, 1, -2, 3, -4, 0], gamma=0.5),
                   [0.0625, 0.125, 0.25, -1.5, 1.0, -4.0, 0.0])
assert np.allclose(get_cumulative_rewards([0, 0, 1, 2, 3, 4, 0], gamma=0), [0, 0, 1, 2, 3, 4, 0])

Czas nauczyć agenta gry w środowisku *CartPool*:

In [6]:
def generate_session(t_max=1000):
    """play env with REINFORCE agent and train at the session end"""

    reward = 0

    reset_out = env.reset()
    s = reset_out[0] if isinstance(reset_out, tuple) else reset_out

    for t in range(t_max):

        # chose action
        a = agent.get_action(s)

        step_out = env.step(a)
        if len(step_out) == 5:
            new_s, r, terminated, truncated, info = step_out
            done = terminated or truncated
        else:
            new_s, r, done, info = step_out

        # record session history to train later
        agent.remember(s, a, r)

        reward += r

        s = new_s
        if done:
            break

    agent.replay()

    return reward


for i in range(100):

    rewards = [generate_session() for _ in range(100)]  # generate new sessions

    print("mean reward:%.3f" % (np.mean(rewards)))

    if np.mean(rewards) > 300:
        print("You Win!")
        break

mean reward:23.140
mean reward:68.920
mean reward:276.500
mean reward:243.610
mean reward:175.030
mean reward:171.310
mean reward:773.840
You Win!
